In [12]:

# ============================ CELL 1 — env + GPU pin ============================
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"          # single T4: no NCCL/device-split hangs
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"




In [13]:
# ============================ CELL 2 — installs ============================
!pip install -q -U "transformers>=4.46" "trl>=0.21" "peft>=0.13" "accelerate>=1.0" \
    "datasets>=3.0" "bitsandbytes>=0.45" hf_transfer --no-cache-dir


In [14]:
import importlib
for pkg in ["torch", "transformers", "trl", "peft", "accelerate", "bitsandbytes", "datasets"]:
    mod = importlib.import_module(pkg)
    print(f"{pkg:15s} {getattr(mod, '__version__', '?')}")

import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")

torch           2.10.0+cu128
transformers    5.12.1
trl             1.6.0
peft            0.19.1
accelerate      1.14.0
bitsandbytes    0.49.2
datasets        5.0.0
CUDA available: True
GPU: Tesla T4


In [15]:
# ============================ CELL 3 — imports + HF auth (Kaggle Secrets) ============================
import torch, random, gc, glob, shutil, psutil
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")

hf_token = None
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass

from huggingface_hub import login, notebook_login
login(token=hf_token) if hf_token else notebook_login()



GPU: Tesla T4


In [16]:
# ============================ CELL 4 — config ============================
DATA_FILE = "/kaggle/input/datasets/mrkrishna225/trainingdata-rp1/dataset_20260622T120406Z.jsonl"
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
MAX_LENGTH = 1536
REFUSAL_RATIO = 0.20
EVAL_FRACTION = 0.05
SEED = 42
random.seed(SEED)

SYSTEM_PROMPT = (
    "You are a strict, factual answering assistant. "
    "Answer the question using ONLY the information explicitly stated in the provided context. "
    "If the context does not contain enough information, respond with exactly: "
    "\"I don't know based on the provided context.\" Never guess."
)
REFUSAL_ANSWER = "I don't know based on the provided context."

RAM_LIMIT_GB = psutil.virtual_memory().total / 1e9 * 0.85   # was undefined before -> crashed training
DISK_LIMIT_GB = shutil.disk_usage("/").total / 1e9

def mem_report(tag=""):
    vm = psutil.virtual_memory()
    used_gb = vm.used / 1e9
    gpu_gb = torch.cuda.memory_allocated() / 1e9 if torch.cuda.is_available() else 0.0
    free_disk_gb = shutil.disk_usage("/").free / 1e9
    print(f"[mem {tag}] RAM {used_gb:.1f}/{vm.total/1e9:.1f}GB | GPU {gpu_gb:.1f}GB | disk free {free_disk_gb:.1f}GB")
    return used_gb, free_disk_gb

def free_mem():
    gc.collect()
    torch.cuda.empty_cache()

# Resolve dataset path on Kaggle (uploaded dataset lives under /kaggle/input/.../)
def resolve_data_path(filename):
    if os.path.exists(filename):
        return filename
    hits = glob.glob(f"/kaggle/input/**/{filename}", recursive=True)
    if hits:
        return hits[0]
    raise FileNotFoundError(f"{filename} not found in cwd or /kaggle/input/* — attach it as a Kaggle dataset.")

DATA_FILE = resolve_data_path(DATA_FILE)
mem_report("startup")



[mem startup] RAM 4.6/33.7GB | GPU 1.9GB | disk free 1179.8GB


(4.566007808, 1179.77888768)

In [17]:
# ============================ CELL 5 — load + clean ============================
from datasets import load_dataset, concatenate_datasets

raw = load_dataset("json", data_files=DATA_FILE, split="train")

def is_valid_row(ex):
    q, a, pos = ex.get("question"), ex.get("answer"), ex.get("positive_ctx")
    return bool(q and a and pos and q.strip() and a.strip() and pos.strip())

cleaned = raw.filter(is_valid_row)

seen, keep_idx = set(), []
for i, ex in enumerate(cleaned):
    key = (ex["question"].strip().lower(), ex["answer"].strip().lower())
    if key not in seen:
        seen.add(key)
        keep_idx.append(i)
cleaned = cleaned.select(keep_idx)
print(f"Rows after cleaning: {len(cleaned)}")
mem_report("after clean")




Rows after cleaning: 2227
[mem after clean] RAM 4.6/33.7GB | GPU 1.9GB | disk free 1179.8GB


(4.565745664, 1179.778895872)

In [18]:
# ============================ CELL 6 — RAFT formatting (prompt/completion, NOT flat text) ============================
# Using explicit prompt/completion columns guarantees the loss mask only covers
# the answer, regardless of whether the model's chat template supports the
# {% generation %} tags assistant_only_loss needs. This is the actual fix for
# "irrelevant answers" — completion_only_loss is True by default for this format.

def to_answerable(ex):
    contexts = [ex["positive_ctx"]] + list(ex.get("negative_ctxs", []) or [])
    random.shuffle(contexts)
    user_msg = "Context:\n" + "\n\n".join(contexts) + f"\n\nQuestion: {ex['question']}"
    return {
        "prompt": [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user_msg}],
        "completion": [{"role": "assistant", "content": ex["answer"]}],
    }

def to_refusal(ex):
    user_msg = "Context:\n" + "\n\n".join(ex["negative_ctxs"]) + f"\n\nQuestion: {ex['question']}"
    return {
        "prompt": [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user_msg}],
        "completion": [{"role": "assistant", "content": REFUSAL_ANSWER}],
    }

drop_cols = cleaned.column_names
answerable_ds = cleaned.map(to_answerable, remove_columns=drop_cols)

has_negs = cleaned.filter(lambda ex: bool(ex.get("negative_ctxs")))
n_refusal = min(int(len(answerable_ds) * REFUSAL_RATIO), len(has_negs))
refusal_idx = random.sample(range(len(has_negs)), n_refusal) if n_refusal > 0 else []
refusal_ds = has_negs.select(refusal_idx).map(to_refusal, remove_columns=has_negs.column_names)
print(f"Added {len(refusal_ds)} refusal examples ({REFUSAL_RATIO*100:.0f}% target).")

formatted_dataset = concatenate_datasets([answerable_ds, refusal_ds]).shuffle(seed=SEED)
del answerable_ds, refusal_ds, has_negs, cleaned, raw
free_mem()
mem_report("after format")


Map:   0%|          | 0/445 [00:00<?, ? examples/s]

Added 445 refusal examples (20% target).
[mem after format] RAM 4.5/33.7GB | GPU 1.9GB | disk free 1179.8GB


(4.518449152, 1179.777613824)

In [19]:
# ============================ CELL 7 — tokenizer + length filter ============================
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.truncation_side = "left"

def add_length(ex):
    n = len(tokenizer.apply_chat_template(ex["prompt"] + ex["completion"], tokenize=True))
    return {"token_length": n}

formatted_dataset = formatted_dataset.map(add_length)
lengths = sorted(formatted_dataset["token_length"])
n = len(lengths)
print(f"tokens — min {lengths[0]} p50 {lengths[n//2]} p90 {lengths[int(n*0.9)]} max {lengths[-1]}")

before = len(formatted_dataset)
formatted_dataset = formatted_dataset.filter(lambda ex: ex["token_length"] <= MAX_LENGTH).remove_columns(["token_length"])
print(f"Dropped {before - len(formatted_dataset)} rows over MAX_LENGTH={MAX_LENGTH}")

split = formatted_dataset.train_test_split(test_size=EVAL_FRACTION, seed=SEED)
train_dataset, eval_dataset = split["train"], split["test"]
print(f"Train: {len(train_dataset)} | Eval: {len(eval_dataset)}")
mem_report("after split")



Map:   0%|          | 0/2672 [00:00<?, ? examples/s]

tokens — min 2 p50 2 p90 2 max 2


Filter:   0%|          | 0/2672 [00:00<?, ? examples/s]

Dropped 0 rows over MAX_LENGTH=1536
Train: 2538 | Eval: 134
[mem after split] RAM 4.5/33.7GB | GPU 1.9GB | disk free 1179.8GB


(4.497117184, 1179.76815616)

In [20]:
# ============================ CELL 8 — model + QLoRA (single T4) ============================
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import torch # Ensure torch is imported if not already in your notebook

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,   # T4 = no bf16 tensor cores
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map={"": 0},                     # pinned single GPU, not "auto"
    torch_dtype=torch.float16,              # Note: changed 'dtype' to 'torch_dtype' for HuggingFace standard, though kwargs usually pass it through
    attn_implementation="sdpa",
    low_cpu_mem_usage=True,
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, peft_config)

# --- START OF RECOMMENDATION FIX ---
# Force every trainable (LoRA) param to fp16 — peft can leave these in
# bfloat16 (inherited from the checkpoint's config.json torch_dtype) even
# when the base model and bnb compute dtype are float16. Mixing that with
# fp16=True's GradScaler crashes, since GradScaler has no bf16 kernel.
for p in model.parameters():
    if p.requires_grad and p.dtype != torch.float16:
        p.data = p.data.to(torch.float16)

# Confirm it worked before training
bad = [n for n, p in model.named_parameters() if p.requires_grad and p.dtype != torch.float16]
print("non-fp16 trainable params:", bad or "none — good")
# --- END OF RECOMMENDATION FIX ---

model.print_trainable_parameters()
mem_report("after model+lora")
free_mem()

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

non-fp16 trainable params: none — good
trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820
[mem after model+lora] RAM 6.6/33.7GB | GPU 3.5GB | disk free 1179.8GB


In [21]:
# ============================ CELL 9 — trainer ============================
from trl import SFTTrainer, SFTConfig
from transformers import TrainerCallback

class MemWatchdog(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        used_gb, free_disk_gb = mem_report(f"step {state.global_step}")
        if used_gb > RAM_LIMIT_GB:
            free_mem()
        if free_disk_gb < (DISK_LIMIT_GB * 0.1):
            print("WARNING: low disk — stopping training.")
            control.should_training_stop = True
        return control

training_args = SFTConfig(
    output_dir="/kaggle/working/qwen-raft-finetune",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    optim="paged_adamw_32bit",
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    num_train_epochs=3,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    fp16=True,
    bf16=False,
    max_length=MAX_LENGTH,
    packing=False,
    completion_only_loss=True,          # <-- THE FIX: loss only on the answer tokens
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    dataloader_num_workers=0,           # avoids RAM duplication from forking after model load
    report_to="none",
    max_grad_norm=0.3,
    seed=SEED,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    args=training_args,
    callbacks=[MemWatchdog()],
)

# ---- sanity check: confirm loss is masked to the answer ONLY, before burning compute ----
sample = trainer.data_collator([trainer.train_dataset[0]])
visible = [t for t in sample["labels"][0].tolist() if t != -100]
print("LOSS-VISIBLE TOKENS (must be ONLY the answer):", repr(tokenizer.decode(visible)))




[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
/tmp/ipykernel_120/3857735558.py:15: FutureWarning: The default `loss_type` will change from `'nll'` to `'chunked_nll'` in TRL 1.7. For standard models this is transparent (same math, lower memory) and no action is needed — you'll get the new default automatically on upgrade. If you use a custom model, check ahead of time that `loss_type='chunked_nll'` runs and yields the same loss as `'nll'`; if it doesn't, pin `loss_type='nll'` to keep the current behavior and please open an issue at https://github.com/huggingface/trl/issues so we can address the edge case.
  training_args = SFTConfig(


Tokenizing train dataset:   0%|          | 0/2538 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/134 [00:00<?, ? examples/s]

LOSS-VISIBLE TOKENS (must be ONLY the answer): 'Events help organizations better understand the state of their applications and systems, spot trends, and handle incidents.<|im_end|>\n'


In [ ]:
# ============================ CELL 10 — train ============================
# --- REPLACE THE PREVIOUS SWEEP WITH THIS ---
# The GradScaler requires trainable parameters (and their gradients) 
# to be in float32 to maintain precision during the optimizer step.
for name, param in model.named_parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.float32)
# --------------------------------------------

print("Starting training...")
mem_report("pre-train")
trainer.train()
mem_report("post-train")

SAVE_DIR = "/kaggle/working/qwen-raft-finetune/final_adapter"
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f"Saved adapter to {SAVE_DIR}")

del trainer, train_dataset, eval_dataset, formatted_dataset
free_mem()
mem_report("after cleanup")


[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Starting training...
[mem pre-train] RAM 6.6/33.7GB | GPU 3.6GB | disk free 1179.7GB


Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
50,0.684634,0.629267,0.649480,0.831439,592203.000000


[mem step 10] RAM 6.6/33.7GB | GPU 3.6GB | disk free 1179.7GB
[mem step 20] RAM 6.6/33.7GB | GPU 3.6GB | disk free 1179.7GB
[mem step 30] RAM 6.6/33.7GB | GPU 3.6GB | disk free 1179.7GB
[mem step 40] RAM 6.6/33.7GB | GPU 3.6GB | disk free 1179.7GB
[mem step 50] RAM 6.6/33.7GB | GPU 3.6GB | disk free 1179.7GB
[mem step 50] RAM 6.6/33.7GB | GPU 3.6GB | disk free 1179.7GB


In [ ]:
# ============================ CELL 11 — merge (fp16, matches training precision) ============================
from peft import PeftModel

del model
free_mem()

base_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float16, low_cpu_mem_usage=True)
merged_model = PeftModel.from_pretrained(base_model, SAVE_DIR).merge_and_unload()

MERGED_DIR = "/kaggle/working/qwen-raft-merged"
merged_model.save_pretrained(MERGED_DIR)
tokenizer.save_pretrained(MERGED_DIR)
print(f"Merged model saved to {MERGED_DIR} — auto-persisted in the Kaggle notebook's Output tab.")

PUSH_TO_HUB = False   # set True + fill in your repo id to also push
if PUSH_TO_HUB:
    hf_repo_id = "mrkrishna1234/Qwen-Chatbot-Merged"
    merged_model.push_to_hub(hf_repo_id)
    tokenizer.push_to_hub(hf_repo_id)
    print(f"Live at https://huggingface.co/{hf_repo_id}")



In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

# 0. Authenticate with Hugging Face using Kaggle Secrets
# (Make sure you have added a secret named "HF_TOKEN" with your write token in Kaggle)
try:
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
    login(token=hf_token)
except Exception as e:
    print("Warning: Could not load HF_TOKEN from Kaggle secrets. Make sure it's set up!")
    # Alternatively, you can paste your token directly here for testing:
    # login(token="hf_your_write_token_here")

hf_repo_id = "mrkrishna1234/Qwen-Chatbot-Merged"

# Assuming these were defined earlier, but placing here for safety. 
# Update them if they are named differently in your notebook!
# model_id = "Qwen/Qwen2.5-..." 
# SAVE_DIR = "./your_adapter_save_path"
# tokenizer = AutoTokenizer.from_pretrained(model_id)

print("1. Loading base model in fast 16-bit precision...")
base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,   # Fixed parameter name and matched to T4 precision
    low_cpu_mem_usage=True,
    device_map="auto"            # Recommended to prevent CPU RAM Out-Of-Memory errors during load
)

print("2. Fusing your trained RAFT adapters into the base model...")
merged_model = PeftModel.from_pretrained(base_model, SAVE_DIR)
merged_model = merged_model.merge_and_unload()

print(f"3. Pushing fully assembled model to Hugging Face ({hf_repo_id})...")
merged_model.push_to_hub(hf_repo_id)
tokenizer.push_to_hub(hf_repo_id)

print(f"✅ Success! Your standalone model is now live at: https://huggingface.co/{hf_repo_id}")

In [ ]:
# ============================ CELL 12 — chat (bounded memory) ============================
from transformers import TextStreamer

merged_model.to("cuda:0")
merged_model.eval()
merged_model.config.use_cache = True
mem_report("pre-inference")

MAX_HISTORY_TURNS = 6

def chat_with_bot():
    print("Custom RAFT Chatbot — 'quit' to exit, 'reset' to clear memory.\n")
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    while True:
        user_input = input("\nYou: ").strip()
        if user_input.lower() in ("quit", "exit"):
            break
        if user_input.lower() == "reset":
            messages = [{"role": "system", "content": SYSTEM_PROMPT}]
            print("(memory cleared)")
            continue
        if not user_input:
            continue

        messages.append({"role": "user", "content": user_input})
        if len(messages) > 1 + MAX_HISTORY_TURNS * 2:
            messages = [messages[0]] + messages[-MAX_HISTORY_TURNS * 2:]

        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        model_inputs = tokenizer([text], return_tensors="pt").to(merged_model.device)
        streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

        print("AI: ", end="")
        with torch.no_grad():
            generated_ids = merged_model.generate(
                **model_inputs, max_new_tokens=512, streamer=streamer,
                do_sample=False, num_beams=1, repetition_penalty=1.1,
                pad_token_id=tokenizer.eos_token_id,
            )
        generated_ids = [out[len(inp):] for inp, out in zip(model_inputs.input_ids, generated_ids)]
        response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()
        if len(response) < 3:
            response = REFUSAL_ANSWER
            print(response)
        messages.append({"role": "assistant", "content": response})

chat_with_bot()